**Imports , NLKT Setup and Query Tokenizer (From Part 1)**


In [1]:
import pandas as pd
import numpy as np
import json
import math
from collections import defaultdict, Counter
import re, unicodedata
import os
import sys 

# NLTK Components for Tokenization 
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords

_STEM = PorterStemmer()
_STOP = set(stopwords.words("english"))
_PUNCT = re.compile(r"[^\w\s]+", re.UNICODE)

def _norm(s: str) -> str:
    """Normalization utility from Part 1 (used for categorical/numeric fields)."""
    if not isinstance(s, str): return ""
    s = unicodedata.normalize("NFKC", s).lower()
    s = re.sub(r"[^\w\s]+", " ", s)
    return re.sub(r"\s+", " ", s).strip()

def build_terms(text: str) -> list[str]:
    """Applies Part 1 preprocessing to the query (tokenize, stem, filter)."""
    if not isinstance(text, str): return []
    s = unicodedata.normalize("NFKC", text.lower())
    s = _PUNCT.sub(" ", s)
    toks = [t for t in s.split() if t not in _STOP]
    toks = [_STEM.stem(t) for t in toks]
    # Retiene caracteres individuales no numéricos (fix para H&M)
    return [t for t in toks if not t.isdigit()]

def _as_tokens(x):
    if isinstance(x, list):
        return x
    if isinstance(x, np.ndarray):
        return x.tolist()
    if x is None or (isinstance(x, float) and math.isnan(x)):
        return []
    if isinstance(x, str):
        # try JSON list first, else whitespace split
        try:
            v = json.loads(x)
            return v if isinstance(v, list) else [w for w in v.split() if w]
        except Exception:
            return [w for w in x.split() if w]
    return []

**Inverted Index Building**

In [2]:
class InvertedIndex:
    """Stores DF, Posting List (pid, tf), and Document Length (L_d)."""
    def __init__(self):
        # Index: { term: { 'df': int, 'postings': { pid: tf } } }
        self.index = defaultdict(lambda: {'df': 0, 'postings': {}})
        # L_d: { pid: L_d } (Euclidean norm for VSM/Cosine Similarity)
        self.doc_lengths = {}
        self.num_docs = 0

    def add_document(self, doc_id: str, tokens: list[str]):
        """Calculates TF and L_d, and adds terms to postings."""
        tf_counts = Counter(tokens)
        
        # L_d (Euclidean norm of the raw term vector)
        L_d = math.sqrt(sum(tf_counts[term]**2 for term in tf_counts))
        self.doc_lengths[doc_id] = L_d
        
        # Update postings list and document frequency (df)
        for term, tf in tf_counts.items():
            if doc_id not in self.index[term]['postings']:
                self.index[term]['df'] += 1
            self.index[term]['postings'][doc_id] = tf
        
        self.num_docs += 1

    def build_from_dataframe(self, df: pd.DataFrame):
        """Builds index from Part 1 processed data."""
        self.num_docs = len(df)
        
        for _, row in df.iterrows():
            pid = row['pid']
            title = _as_tokens(row['title_tokens'])
            desc  = _as_tokens(row['desc_tokens'])
            details = _as_tokens(row.get('details_tokens', []))
            tokens = title + desc + details
            self.add_document(pid, tokens)
        
        print(f"Index built with {self.num_docs} documents and {len(self.index)} terms.")
    
    def get_term_stats(self, term):
        """Retrieves statistics for a given term."""
        return self.index.get(term, {'df': 0, 'postings': {}})

**TF-IDF Ranking and Retrieval**

In [3]:
def calculate_tfidf_weight(tf, df, N):
    """Calculates the W_t,d or W_t,q TF-IDF weight."""
    # TF Component: 1 + log(tf) (Log-frequency weighting)
    tf_comp = 1 + math.log10(tf) if tf > 0 else 0
    # IDF Component: log(N/df)
    idf_comp = math.log10(N / df) if df > 0 else 0
    return tf_comp * idf_comp

def ranked_search(query: str, index: InvertedIndex) -> list[tuple[str, float]]:
    N = index.num_docs
    terms = build_terms(query)
    if not terms:
        return []

    # AND retrieval
    doc_sets = []
    for t in terms:
        postings = index.get_term_stats(t)['postings']
        if not postings:
            return []  # any missing term -> empty
        doc_sets.append(set(postings.keys()))
    retrieved = list(set.intersection(*doc_sets))

    # Cosine with weighted denominator (over query terms)
    q_tf = Counter(terms)
    scores = {}
    for pid in retrieved:
        num = 0.0
        den = 0.0
        for t in terms:
            stats = index.get_term_stats(t)
            df = stats['df']
            w_d = calculate_tfidf_weight(stats['postings'].get(pid, 0), df, N)
            w_q = calculate_tfidf_weight(q_tf[t], df, N)
            num += w_q * w_d
            den += w_d * w_d
        scores[pid] = (num / math.sqrt(den)) if den > 0 else 0.0

    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


**Evaluation Metrics**

In [4]:
def get_relevance_labels(query_id, retrieved_pids, df_labels):
    """Returns ranked binary relevance scores (1/0) and R_total."""
    relevant_labels_df = df_labels[df_labels['query_id'] == query_id]
    relevant_pids = set(relevant_labels_df[relevant_labels_df['relevance'] == 1]['pid'])
    R_total = len(relevant_pids)
    relevance_scores = [1 if pid in relevant_pids else 0 for pid in retrieved_pids]
    return relevance_scores, R_total

# Standard Cutoff Metrics 
def precision_at_k(rel_scores, k):
    """P@K (Required i)"""
    if k == 0 or not rel_scores: return 0.0
    k = min(k, len(rel_scores))
    return sum(rel_scores[:k]) / k

def recall_at_k(rel_scores, R_total, k):
    """R@K (Required ii)"""
    if R_total == 0: return 0.0
    k = min(k, len(rel_scores))
    return sum(rel_scores[:k]) / R_total

def f1_score_at_k(P_at_k, R_at_k):
    """F1-Score@K (Required iv)"""
    if P_at_k + R_at_k == 0: return 0.0
    return 2 * P_at_k * R_at_k / (P_at_k + R_at_k)

# --- Ranking Metrics ---
def average_precision_at_k(rel_scores, k):
    """AP@K (Required iii)"""
    if not rel_scores: return 0.0
    k = min(k, len(rel_scores))
    sum_of_precisions = 0.0
    num_relevant = 0
    for i in range(k):
        if rel_scores[i] == 1:
            num_relevant += 1
            sum_of_precisions += num_relevant / (i + 1)
    return sum_of_precisions / num_relevant if num_relevant > 0 else 0.0

def mean_average_precision(aps_list):
    """MAP (Required v)"""
    return sum(aps_list) / len(aps_list) if aps_list else 0.0

def mean_reciprocal_rank(rel_scores: list):
    """MRR (Required vi)"""
    for i, rel in enumerate(rel_scores):
        if rel == 1:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(rel_scores, k):
    """NDCG@K (Required vii)"""
    k = min(k, len(rel_scores))
    
    # DCG (Actual Ranking)
    dcg = sum(rel_scores[i] / math.log2(i + 2) for i in range(k))
    
    # IDCG (Ideal Ranking)
    ideal_scores = sorted(rel_scores, reverse=True)
    idcg = sum(ideal_scores[i] / math.log2(i + 2) for i in range(k))
        
    return dcg / idcg if idcg > 0.0 else 0.0

**Execution - Build Index and Define Queries**

In [5]:
from pathlib import Path
# 1) Define base directory
BASE = Path(".").resolve() 
# 2) Define files directly
PROCESSED_DATA_FILE = BASE / r"data\processed\products_clean.parquet"
LABELS_FILE         = BASE / r"data\raw\validation_labels.csv"
INDEX_FILE          = BASE / r"data\index\inverted_index.json"

# 3) Sanity checks
print("processed:", PROCESSED_DATA_FILE.exists(), PROCESSED_DATA_FILE)
print("labels   :", LABELS_FILE.exists(), LABELS_FILE)
os.makedirs(INDEX_FILE.parent, exist_ok=True)

# 4) Load data 
try:
    df_clean = pd.read_parquet(PROCESSED_DATA_FILE)
except Exception as e:
    print("pandas read_parquet failed:", e)
    import pyarrow.dataset as ds
    df_clean = ds.dataset(PROCESSED_DATA_FILE, format="parquet").to_table().to_pandas()

# 5) Minimal schema check
need_cols = {"pid","title_tokens","desc_tokens"}
missing = need_cols - set(df_clean.columns)
if missing:
    raise ValueError(f"Missing columns: {missing}")

# 6) Build index
INDEX = InvertedIndex()
INDEX.build_from_dataframe(df_clean)
with open(INDEX_FILE, "w") as f:
    json.dump({"index": dict(INDEX.index), "doc_lengths": dict(INDEX.doc_lengths)}, f)
print(f"Index saved: {INDEX_FILE} | docs={INDEX.num_docs}, terms={len(INDEX.index)}")

processed: True C:\Users\david\IRWA_Definitivo\IRWA_Final_Project\project_progress\part_2\data\processed\products_clean.parquet
labels   : True C:\Users\david\IRWA_Definitivo\IRWA_Final_Project\project_progress\part_2\data\raw\validation_labels.csv
pandas read_parquet failed: Repetition level histogram size mismatch


OSError: Repetition level histogram size mismatch

**Load Labels**

In [ ]:
# Load schema
df_labels = pd.read_csv(LABELS_FILE, dtype={'query_id': int, 'pid': str})
if 'labels' in df_labels.columns and 'relevance' not in df_labels.columns:
    df_labels = df_labels.rename(columns={'labels': 'relevance'})
if 'title' in df_labels.columns:
    df_labels = df_labels.drop(columns=['title'])

# Enforce types and values
df_labels = df_labels.astype({'query_id': int, 'pid': str, 'relevance': int})

# Sanity check
need = {"query_id", "pid", "relevance"}
missing = need - set(df_labels.columns)
if missing:
    raise ValueError(f"validation_labels.csv missing columns: {missing}")
print("labelled query_ids:", sorted(df_labels["query_id"].unique().tolist()))
# show first 3 rows
print(df_labels.head(3))


**Show most frequent terms for building new queries**

In [ ]:
term_freqs = {term: len(stats['postings']) for term, stats in INDEX.index.items()}
top_terms = sorted(term_freqs.items(), key=lambda x: x[1], reverse=True)[:30]
print(top_terms)

**Define Queries**

In [ ]:
ALL_QUERIES = [
    {'query_id': 1, 'query': "women full sleeve sweatshirt cotton"},
    {'query_id': 2, 'query': "men slim jeans blue"},
    {'query_id': 3, 'query': "long sleeve denim jacket blue"},
    {'query_id': 4, 'query': "cotton shirt man regular fit"},
    {'query_id': 5, 'query': "women western wear cotton"},
    {'query_id': 6, 'query': "machine wash suitabl woman"},
    {'query_id': 7, 'query': "brand blend fabric shirt"},
]
K = 10


**Ranking Results for Queries 1 and 2**

In [ ]:
# top 10 results for each query
def show_topk(q, k=10):
    ranked = ranked_search(q, INDEX)[:k]
    df = df_clean.set_index('pid')
    rows = []
    for pid, s in ranked:
        title = df.loc[pid, 'title_tokens'] if pid in df.index else ''
        rows.append({'rank': len(rows)+1, 'pid': pid, 'title': title, 'score': round(s,3)})
    return pd.DataFrame(rows)

for q in [
    "women full sleeve sweatshirt cotton",
    "men slim jeans blue",
]:
    print(q); display(show_topk(q))

# worked out example token
term = "sweatshirt"; stats = INDEX.get_term_stats(term)
df_t = stats['df']; N = INDEX.num_docs
pid = ranked_search("women full sleeve sweatshirt cotton", INDEX)[0][0]
tf_d = stats['postings'].get(pid, 0)
tf_q = Counter(build_terms("women full sleeve sweatshirt cotton"))[term]
idf = math.log10(N/df_t); w_d = (1+math.log10(tf_d))*idf; w_q = (1+math.log10(tf_q))*idf
contrib = w_d*w_q
print({"term":term,"df":df_t,"idf":idf,"tf_d":tf_d,"w_d":w_d,"tf_q":tf_q,"w_q":w_q,"dot_contrib":contrib})


**Evaluation of Ranking Results for Queries 1 and 2**

In [ ]:
df_labels['pid'] = df_labels['pid'].str.upper()
df_clean['pid'] = df_clean['pid'].str.upper()

K = 10

# Only queries 1 and 2
EVAL_QUERIES = [q for q in ALL_QUERIES if q['query_id'] in (1, 2)]

print("\n--- 2. RUNNING RANKED SEARCH AND EVALUATION (Q1–Q2) ---")
RESULTS, AP_SCORES = [], []

for q in EVAL_QUERIES:
    qid, qtext = q['query_id'], q['query']
    ranked = ranked_search(qtext, INDEX)
    retrieved_pids = [pid for pid, _ in ranked]

    rel_scores, R_total = get_relevance_labels(qid, retrieved_pids, df_labels)

    Pk   = precision_at_k(rel_scores, K)
    Rk   = recall_at_k(rel_scores, R_total, K)
    F1k  = f1_score_at_k(Pk, Rk)
    APk  = average_precision_at_k(rel_scores, K)
    MRR  = mean_reciprocal_rank(rel_scores)
    NDCG = ndcg_at_k(rel_scores, K)

    AP_SCORES.append(APk)
    RESULTS.append({
        'Query ID': qid, 'Query Text': qtext, 'R_Total': R_total,
        'Retrieved': len(retrieved_pids), 'P@10': round(Pk,3),
        'R@10': round(Rk,3), 'F1@10': round(F1k,3),
        'AP@10': round(APk,3), 'MRR': round(MRR,3), 'NDCG@10': round(NDCG,3)
    })

df_results = pd.DataFrame(RESULTS)
print(df_results.to_string(index=False))
print(f"\nMAP over {len(EVAL_QUERIES)} queries: {round(mean_average_precision(AP_SCORES), 3)}")



**Ground Truth for New Queries**

In [ ]:
test_queries =[
    "long sleeve denim jacket blue",
    "cotton shirt man regular fit",
    "women western wear cotton",
    "machine wash suitabl woman",
    "brand blend fabric shirt"
]
for q in test_queries: 
    print(q); display(show_topk(q))


In [ ]:
manual_judgments = [
    # Query 3 — long sleeve denim jacket blue
    {"query": "long sleeve denim jacket blue", "pid": "SHTFRVPAV33BCXEZ", "relevance": 0},  # Shirt no jacket (shirt ≠ jacket)
    {"query": "long sleeve denim jacket blue", "pid": "JCKFT9TE2DB6TJNQ", "relevance": 1},  # Denim jaquet
    {"query": "long sleeve denim jacket blue", "pid": "JCKFPEVFF5DWZB4H", "relevance": 1},  # Denim jacket
    {"query": "long sleeve denim jacket blue", "pid": "JCKFKKFZMDEHEZQS", "relevance": 1},  # Denim jaquet, relevant
    {"query": "long sleeve denim jacket blue", "pid": "JCKFKKFZ7QB2KUQV", "relevance": 1},  # Blue denim jaquet, relevant
    {"query": "long sleeve denim jacket blue", "pid": "JCKFEED3BHMKFAJY", "relevance": 1},  # Denim jacket, relevant
    {"query": "long sleeve denim jacket blue", "pid": "JCKFS4NFYXFDRWDH", "relevance": 1},  # Casual men jaquet relevant, even if its not denim matches quite well.

    # Query 4: "cotton shirt man regular fit"
    {"query": "cotton shirt man regular fit", "pid": "TSHFNV35W6XMETKT", "relevance": 0},  # Has "print" and colors, no mention of cotton or regular fit.
    {"query": "cotton shirt man regular fit", "pid": "TSHFU684YFWES5FZ", "relevance": 0},  # Printed shirt, no cotton or regular fit mentioned.
    {"query": "cotton shirt man regular fit", "pid": "TSHFJ9MCTHXXVUF9", "relevance": 0},  # Printed design, not clearly cotton or regular fit.
    {"query": "cotton shirt man regular fit", "pid": "TSHFUDHTG2AMH5SH", "relevance": 0},  # Graphic print shirt, lacks reference to cotton or regular fit.
    {"query": "cotton shirt man regular fit", "pid": "TSHFNV9K3N5BAQDR", "relevance": 0},  # Superhero print shirt, not cotton or regular fit.
    {"query": "cotton shirt man regular fit", "pid": "TSHFNV9K2RBPGQH4", "relevance": 0},  # Same as above, printed style not relevant.
    {"query": "cotton shirt man regular fit", "pid": "TSHFNV48JMW4GYH", "relevance": 0},   # Printed shirt, no cotton or regular fit information.
    {"query": "cotton shirt man regular fit", "pid": "TSHFJ8UADC9KVTVC", "relevance": 0},  # Printed shirt, lacks cotton and regular fit terms.
    {"query": "cotton shirt man regular fit", "pid": "TSHFWT9SH7T2H2KW", "relevance": 0},  # Printed shirt, not described as cotton or regular fit.
    {"query": "cotton shirt man regular fit", "pid": "SHTFT26G9JRR3RZF", "relevance": 1},  # Contains "men, regular fit, solid", matches the query well unstead not having cotton.

    # Query 5: "women western wear cotton"
    {"query": "women western wear cotton", "pid": "SHTEMD8QK7HNEFS8", "relevance": 1},  # “Women, regular fit, solid, collar” – fits cotton and western wear style.
    {"query": "women western wear cotton", "pid": "SHTUFNBH4M46HSV", "relevance": 1},   # “Women, slim fit, solid, mandarin collar” – suitable for western wear.
    {"query": "women western wear cotton", "pid": "SHTFPS2FHWZHX9ZE", "relevance": 1},  # “Women, slim fit, checker, mandarin collar” – plausible for western wear.
    {"query": "women western wear cotton", "pid": "SHTFYRH2WRZEUHT", "relevance": 1},   # “Women, slim fit, print, casual shirt” – fits cotton/casual western style.
    {"query": "women western wear cotton", "pid": "SHTFRR7FHGVAZHTB", "relevance": 1},  # Similar to above, relevant casual shirt.
    {"query": "women western wear cotton", "pid": "SHTFYTWBJTGEWTFB", "relevance": 1},  # “Women, slim fit, solid, spread collar” – relevant to the query.
    {"query": "women western wear cotton", "pid": "SHTVGFS2THNCQRH", "relevance": 1},   # “Women, slim fit, casual shirt” – matches the intent.
    {"query": "women western wear cotton", "pid": "SHTFZPVAMZVAHGTJ", "relevance": 1},  # Another casual shirt, consistent with the query.

    # Query 6: "machine wash suitabl woman"
    {"query": "machine wash suitabl woman", "pid": "SHTFVQUR6VHSEPHE", "relevance": 1},  # Women regular fit printed shirt – casual and likely machine washable.
    {"query": "machine wash suitabl woman", "pid": "JCKFMV5X7CFQMNYR", "relevance": 0},  # Sleeveless casual jacket – not a shirt, less suitable for query intent.
    {"query": "machine wash suitabl woman", "pid": "TSHFXCXZSBH5UFJT", "relevance": 1},  # Women round neck white shirt – simple, washable, relevant.
    {"query": "machine wash suitabl woman", "pid": "SHTFWZHNC6NWBQMH", "relevance": 1},  # Slim fit solid button casual shirt – clearly fits a washable women’s shirt.
    {"query": "machine wash suitabl woman", "pid": "TSHFW4S4JVEZWDAM", "relevance": 0},  # Superhero print shirt – thematic, less about wash suitability.
    {"query": "machine wash suitabl woman", "pid": "TSHFNSFPHTAYQYUV", "relevance": 1},  # Graphic print women’s shirt – casual, washable, relevant.
    {"query": "machine wash suitabl woman", "pid": "SHTFVSXRRCNEWMMP", "relevance": 1},  # Regular fit printed collar shirt – likely machine washable.
    {"query": "machine wash suitabl woman", "pid": "SHTFVQUZ43XYCSBN", "relevance": 1},  # Regular fit printed collar shirt – fits casual washable style.
    {"query": "machine wash suitabl woman", "pid": "SHTFWZHC2UYQYFRY", "relevance": 1},  # Slim fit solid casual shirt – suitable for the query intent.
    {"query": "machine wash suitabl woman", "pid": "JCKFMV5WTASYK4Y2", "relevance": 0},  # Full sleeve casual jacket – not a shirt, irrelevant.

    # Query 7: "brand blend fabric shirt"
    {"query": "brand blend fabric shirt", "pid": "DHTFV8UGUQGJGGCG", "relevance": 0},  # Dhoti – completely different product, not a shirt.
    {"query": "brand blend fabric shirt", "pid": "DHTFVYTE5QHBGVHG", "relevance": 0},  # Dhoti – not a shirt or blended fabric product.
    {"query": "brand blend fabric shirt", "pid": "DHTFVYAP4ZWN85ZS", "relevance": 0},  # Dhoti – irrelevant to shirts or fabric blends.
    {"query": "brand blend fabric shirt", "pid": "DHTFVY9TQKQSGGGX", "relevance": 0},  # Dhoti – unrelated garment type.
    {"query": "brand blend fabric shirt", "pid": "DHTFV3KYNSUDGWBB", "relevance": 0},  # Dhoti – not a shirt.
    {"query": "brand blend fabric shirt", "pid": "DHTFV3YFCVHWD7CJ", "relevance": 0},  # Dhoti – different product.
    {"query": "brand blend fabric shirt", "pid": "DHTFMZ9PSTNED8RN", "relevance": 0},  # Dhoti – unrelated to query terms (fabric shirt).
    {"query": "brand blend fabric shirt", "pid": "DHTFV2GYPTKXZUMJ", "relevance": 0},  # Dhoti – not a shirt or brand blend item.
    {"query": "brand blend fabric shirt", "pid": "DHTFMZ9UMHFSBZW3", "relevance": 0},  # Dhoti – irrelevant.
    {"query": "brand blend fabric shirt", "pid": "DHTFV8UGWPM7KRQY", "relevance": 0},  # Dhoti – not a shirt or fabric blend.
]


df_manual = pd.DataFrame(manual_judgments)
df_manual

df_labels = pd.read_csv(LABELS_FILE)

# Combinar los juicios existentes con los nuevos
df_labels_updated = pd.concat([df_labels, df_manual], ignore_index=True)

# Guardar versión actualizada
df_labels_updated.to_csv(LABELS_FILE, index=False)

print(f"✅ Archivo actualizado con {len(df_manual)} nuevos juicios.")
print("Ruta:", LABELS_FILE)


**Inspect top results**

In [ ]:
for q in ALL_QUERIES[2:]:
    print(q['query'])
    ranked = ranked_search(q['query'], INDEX)
    print([pid for pid,_ in ranked[:10]])
    print()
